In [1]:
# Load packages 
import datasets 
datasets.disable_caching()
from transformers import set_seed 
set_seed(42)

/mimer/NOBACKUP/groups/naiss2024-22-903/anaconda3/envs/RL/lib/python3.8/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# Take in a course 
val_dataset = datasets.load_dataset("csv",data_files={"train": "data/train_data.csv"}, split='train')

In [3]:
import pandas as pd 
df = pd.read_excel('data/courses.ods', engine='odf')
print(df)

   University                     Type   Credits   \
0          LU                 Elective        7.5   
1          LU                 Elective        7.5   
2          LU                 Elective        7.5   
3          LU                 Elective        7.5   
4         KTH                Mandatory        4.5   
5         KTH                Mandatory        3.0   
6         KTH                Mandatory        2.0   
7         KTH                Mandatory        7.5   
8         KTH                Mandatory        7.5   
9         KTH                Mandatory        7.5   
10        KTH                Mandatory        7.5   
11        KTH   Conditionally Elective        7.5   
12        KTH   Conditionally Elective        7.5   
13        KTH   Conditionally Elective        7.5   
14        KTH   Conditionally Elective        7.5   
15        KTH   Conditionally Elective        7.5   
16        KTH   Conditionally Elective        7.5   
17        KTH   Conditionally Elective        

In [4]:
df.keys()

Index(['University ', 'Type ', 'Credits ', 'Course name ', 'Course code  ',
       'Learning objectives ', 'Course Description '],
      dtype='object')

In [5]:
course_names = df.get(df.keys()[3])
print(course_names)
course_descriptions = df.get(df.keys()[-1])
LO = df.get(df.keys()[6])
#print(LO)

0                                Advanced Web Security 
1                           Secure Systems Engineering 
2                                         Cryptography 
3                                         Web Security 
4     Theory and Methodology of Science (Natural and...
5     Theory of Science and Scientific methods in Cy...
6        The Cybersecurity Engineer's Role in Society  
7                              Cybersecurity Overview  
8          Cybersecurity in a Socio-Technical Context  
9                                Applied Cryptography  
10                                    Ethical Hacking  
11                         Foundations of Cryptography 
12                     Privacy Enhancing Technologies  
13                  Project course in System Security  
14                            Language-Based Security  
15    Cyber-Physical Security in Time-Critical Syste...
16                         Networked Systems Security  
17                Advanced Networked Systems Sec

In [12]:
from utils.llm_utils import submit_message_LLM, topic_prompt 

In [7]:
# Load Qwen2-7B-Instruct model. 
# gets saved at naiss2024-22-903/.cache/huggingface/models
from transformers import AutoModelForCausalLM, AutoTokenizer
API_KEY="<YOUR-APKI-KEY-HERE>"
model_path = "deepseek-ai/deepseek-r1-distill-qwen-14B" 
# change later to: model_id = "deepseek-ai/DeepSeek-R1
#model_path="Qwen/Qwen2.5-7B-Instruct"
device = "cuda" # the device to load the model onto

model2 = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [8]:
print(course_names)

0                                Advanced Web Security 
1                           Secure Systems Engineering 
2                                         Cryptography 
3                                         Web Security 
4     Theory and Methodology of Science (Natural and...
5     Theory of Science and Scientific methods in Cy...
6        The Cybersecurity Engineer's Role in Society  
7                              Cybersecurity Overview  
8          Cybersecurity in a Socio-Technical Context  
9                                Applied Cryptography  
10                                    Ethical Hacking  
11                         Foundations of Cryptography 
12                     Privacy Enhancing Technologies  
13                  Project course in System Security  
14                            Language-Based Security  
15    Cyber-Physical Security in Time-Critical Syste...
16                         Networked Systems Security  
17                Advanced Networked Systems Sec

In [9]:
idx = 17
print('Description: ',course_descriptions[idx])
description = course_descriptions[idx]
print('Course name: ',course_names[idx]) 
topic = course_names[idx]

Description:  The course covers security including integrity for a spectrum of network systems that includes: Internet and TCP/IP networks, Mobile voice and data networks, wireless local and personal networks, Wireless sensor networks, mobile ad hoc and hybrid networks, such as vehicle communication systems The emphasis of the course is to strengthen the student's understanding of concept and technology, joint security requirements for different systems, how functions in each system determines the latest security solutions and how design decisions should be grasped for efficient security solutions.  
Course name:  Advanced Networked Systems Security  


In [10]:
from utils.load_data import clean_text
message = topic_prompt(topic, description)      # create prompt based on topic + description
device = 'cuda:0'
response = submit_message_LLM(model2, message, tokenizer, device)  # put the formatted prompt through the LLM 
#print('response: ',response)
subtopics = response.split("\n")                # split response into subtopics
for sub in subtopics: 
    #sub = clean_text(sub[2:])
    print(sub)
    #topics.append(sub)                      # add subtopics and the original label to lists 
    #labels.append(label)
    #descriptions.append(description)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Okay, I need to tackle this query where the user is asking for a list of subtopics based on the given description of an advanced course on networked systems security. Let me first understand what the user is looking for. They provided a topic and a detailed description, and they want me to extract all the subtopics without any explanations or additional text.

Looking at the description, the course covers a broad range of network systems, including Internet and TCP/IP networks, Mobile voice and data networks, wireless local and personal networks, Wireless sensor networks, and mobile ad hoc and hybrid networks like vehicle communication systems. The emphasis is on understanding concepts, technologies, joint security requirements, how functions in each system determine the latest solutions, and design decisions for efficient security.

So, I need to break down each part of the description into subtopics. Starting with the different types of networks mentioned: Internet/TCP/IP, Mobile voi

In [37]:
import re
output = response.split('1.')[1] # removes all 'thinking' output 
#print(output)
subtopic = re.split(r'\s\d+\.\s', output)
subtopics = {}
for i, sub in enumerate(subtopic): 
    # if key does not exist, create it, otherwise append? 
    #print(clean_text(sub))
    subtopics[str(i)] = [clean_text(sub)]
print(subtopics)

{'0': [' internet and tcp ip networks,'], '1': ['mobile voice and data networks,'], '2': ['wireless local and personal networks,'], '3': ['wireless sensor networks,'], '4': ['mobile ad hoc and hybrid networks (including vanets),'], '5': ['security concepts and technologies,'], '6': ['joint security requirements,'], '7': ['functions determining security solutions,'], '8': ['design decisions']}


In [38]:
# make them into the right format (a dataset?) 
from datasets import Dataset 
course = Dataset.from_dict(subtopics)
print(course)

Dataset({
    features: ['0', '1', '2', '3', '4', '5', '6', '7', '8'],
    num_rows: 1
})


In [46]:
# Load trained DistilBERT model 
from sklearn.metrics import multilabel_confusion_matrix
from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from transformers import DataCollatorWithPadding
import sys
sys.path.append("/mimer/NOBACKUP/groups/naiss2024-22-903/LLMedu")

# load model from a working checkpoint 
model_name='DistilBERTv1final/checkpoint-430' #'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)             # the tokenizer is the same for all folds 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer) # the data collator is the same for all folds 


In [99]:
import torch
import numpy as np # misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()
llm_labels = [] 
for i in range(len(course.features)): 
    print(course[str(i)])
    tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
        predictions = sigmoid(predictions.logits).detach().cpu().numpy()
        predictions = (predictions > 0.5).astype(int).reshape(-1)
        #if sum(llm_label) == 0: # if we also want to use inputs that are not confident enough 
        #3    llm_label = np.zeros((1,9))
        #    idx = np.argmax(predictions)
        #    llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        llm_labels.append(predictions)
# combine all labels of the course. 
llm_labels = np.array(llm_labels)
print(llm_labels)
final_output = np.sum(llm_labels,axis=0) / (np.sum(llm_labels)) #/ len(course.features) # divide by the total number of 'items' 
print(final_output)
print([KAs[str(j.item())] for j in np.where(final_output>0)[0]])

[' internet and tcp ip networks,']
['mobile voice and data networks,']
['wireless local and personal networks,']
['wireless sensor networks,']
['mobile ad hoc and hybrid networks (including vanets),']
['security concepts and technologies,']
['joint security requirements,']
['functions determining security solutions,']
['design decisions']
[[0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 1 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 1 1]]
[0.  0.  0.  0.  0.6 0.  0.  0.2 0.2]
['connection security', 'organizational security', 'societal security']


### Old code 

In [ ]:
if sum(llm_label) == 0: 
            llm_label = np.zeros((1,9))
            idx = np.argmax(predictions)
            llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        #print(llm_label)
        llm_labels.append(llm_label)
        labels = [KAs[str(j.item())] for j in np.where(llm_label>0)[0]]
        print(labels)
# combine all labels of the course. 
print(llm_labels)

In [79]:
import torch
import numpy as np # misc: (this includes Computer Science, Business and Law, Communication and Networking, Information Technology, Cyberspace Practice, Pedagogy, and Intelligence)
KAs = {"0": "miscellaneous",
           "1": "data security", 
           "2": "software security",
           "3": "component security", 
           "4": "connection security", 
           "5": "system security", 
           "6": "human security", 
           "7": "organizational security",
           "8": "societal security"}

def sigmoid(x):
   return 1/(1 + np.exp(-x))
    
model = AutoModelForSequenceClassification.from_pretrained(model_name, output_hidden_states=True,
                                                                        problem_type="multi_label_classification",
                                                                        num_labels=9, dropout=0.2)
model.eval()
llm_labels = [] 
for i in range(len(course.features)): 
    print(course[str(i)])
    tokenized_course[str(i)] = tokenizer(course[str(i)], return_tensors='pt') #, padding='max_length', max_length=64)
    with torch.no_grad(): 
        predictions= model(**tokenized_course[str(i)])
       # print(predictions)
        predictions = sigmoid(predictions.logits).detach().cpu().numpy()
        #print(predictions)
        llm_label = (predictions > 0.5).astype(int).reshape(-1)
        if sum(llm_label) == 0: 
            llm_label = np.zeros((1,9))
            idx = np.argmax(predictions)
            llm_label[:,idx] = 1 # adjust if there are multiple courses passed at the same time. 
        #print(llm_label)
        llm_labels.append(llm_label)
        labels = [KAs[str(j.item())] for j in np.where(llm_label>0)[0]]
        print(labels)
# combine all labels of the course. 
print(llm_labels)


[' internet and tcp ip networks,']
[[0.0284835  0.02497495 0.00933375 0.01276194 0.90398973 0.02706327
  0.0147735  0.05587436 0.01742662]]
['connection security']
['mobile voice and data networks,']
[[0.03850658 0.29299477 0.00253389 0.00262391 0.17751783 0.04129511
  0.25054374 0.13214242 0.0284964 ]]
['miscellaneous']
['wireless local and personal networks,']
[[0.01354401 0.12714781 0.00312637 0.00329171 0.51857793 0.01852061
  0.13320011 0.16161613 0.0428065 ]]
['connection security']
['wireless sensor networks,']
[[0.02087427 0.10677362 0.00225869 0.00296906 0.62087655 0.03002157
  0.05837579 0.12195189 0.0210558 ]]
['connection security']
['mobile ad hoc and hybrid networks (including vanets),']
[[0.09991411 0.02833556 0.00209423 0.00191664 0.18079734 0.01336825
  0.05636595 0.27459657 0.02014716]]
['miscellaneous']
['security concepts and technologies,']
[[0.11782232 0.02469447 0.01037377 0.00820424 0.01068093 0.26913506
  0.00614757 0.28396702 0.0035062 ]]
['miscellaneous']
['j

In [ ]:
# Then save the course and pass all the descriptions through the trained network 

In [ ]:
# save the outputs 